SHAP summary plot

In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
import optuna

from xgboost import XGBRegressor

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


# =========================
# 1. 读取数据
# =========================
df0 = pd.read_excel("Dataset_DFT.xlsx")
df = df0.copy()

# 如需删除无关列，可取消注释
df.drop(columns=["Type", "DBPs","DOI", "Code"], inplace=True)


# =========================
# 2. 定义输入和输出
# =========================
target_col = "C"

X = df.drop(columns=[target_col])
y = df[target_col]


# =========================
# 3. 定义需要归一化的列
# =========================
numerical_cols_to_scale = [
    "DOC", "UV254", "Br", "TN", "Chlorine dose", "Disinfection time",
    "Temperature", "pH", "C1", "C2", "LMW", "MMW", "HMW",
    "Es", "EHOMO", "ELUMO", "Egap"
]


# =========================
# 4. 先划分训练集和测试集
# =========================
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=72
)


# =========================
# 5. 构建预处理器
# 只归一化指定列，其余列保持不变
# =========================
preprocessor = ColumnTransformer(
    transformers=[
        ("scale", StandardScaler(), numerical_cols_to_scale)
    ],
    remainder="passthrough"
)

# =========================
# 导出完整列，并保持与 train_raw/test_raw 相同的样本顺序
# =========================

# 按 X_train_raw 和 X_test_raw 的实际顺序提取原始完整数据
train_full = df0.loc[X_train_raw.index].copy()
test_full = df0.loc[X_test_raw.index].copy()

# 添加数据集标记
train_full["Dataset_split"] = "Training"
test_full["Dataset_split"] = "Test"

# 先放训练集，再放测试集
dataset_full = pd.concat(
    [train_full, test_full],
    axis=0,
    ignore_index=True
)

dataset_full.to_excel(
    "Dataset_train_test_split.xlsx",
    index=False
)

print("训练集样本数：", len(train_full))
print("测试集样本数：", len(test_full))
print("完整数据已保存至 Dataset_train_test_split.xlsx")

In [ ]:
import shap
import joblib
import pandas as pd
import numpy as np

# =========================
# 1. 读取 Pipeline 模型
# =========================
final_model = joblib.load("final_CDHLQ.pkl")

preprocessor = final_model.named_steps["preprocess"]
xgb_model = final_model.named_steps["model"]


# =========================
# 2. 对训练集进行同样的预处理
# =========================
X_train_processed = preprocessor.transform(X_train_raw)

feature_names = preprocessor.get_feature_names_out()

# 去掉 ColumnTransformer 自动添加的前缀
feature_names = [
    name.replace("scale__", "").replace("remainder__", "")
    for name in feature_names
]

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train_raw.index
)


# =========================
# 3. 创建 SHAP Explainer
# =========================
explainer = shap.TreeExplainer(xgb_model)


# =========================
# 4. 计算 SHAP 值
# =========================
shap_values = explainer(X_train_processed)

# # 可选：计算 SHAP interaction values
# interaction_values = explainer.shap_interaction_values(X_train_processed)


# =========================
# 5. 每个样本的 SHAP 值
# =========================
shap_df = pd.DataFrame(
    shap_values.values,
    columns=X_train_processed.columns,
    index=X_train_processed.index
)

shap_df.insert(0, "Sample_ID", range(1, len(shap_df) + 1))


# =========================
# 6. 平均绝对 SHAP 值
# =========================
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)

importance_df = pd.DataFrame({
    "Feature": X_train_processed.columns,
    "Mean(|SHAP|)": mean_abs_shap
}).sort_values(
    by="Mean(|SHAP|)",
    ascending=False
).reset_index(drop=True)


# =========================
# 7. 导出 Excel
# =========================
with pd.ExcelWriter("SHAP-CDHLQ.xlsx") as writer:
    shap_df.to_excel(writer, sheet_name="SHAP Values", index=False)
    importance_df.to_excel(writer, sheet_name="Feature Importance", index=False)


print("✅ SHAP 计算完成，结果已导出：SHAP-CDHLQ.xlsx")

In [ ]:
import shap
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 15            # 控制坐标轴刻度字体大小
plt.rcParams['axes.titlesize'] = 14       # 控制图标题字号
plt.rcParams['axes.labelsize'] = 20       # 控制轴标签字体大小
plt.rcParams['xtick.labelsize'] = 11      # x轴刻度字体
plt.rcParams['ytick.labelsize'] = 11      # y轴刻度字体

# 构建四段颜色渐变：低 → 中1 → 中2 → 高

custom_cmap = LinearSegmentedColormap.from_list(
     "custom_cmap", ["#867fba", "#86a4cb", "#efddaf", "#c11b24"]
 )


# ✅ 你指定的特征名称
selected_features = [
    'DOC', 'UV254', 'Br', 'TN', 'Chlorine dose', 'Disinfection time',
    'Temperature', 'pH','C1', 'C2', 'LMW', 'MMW', 'HMW',
    'Es','EHOMO','ELUMO','Egap'
]

# 获取在 X_train 中的列索引
selected_indices = [X_train_raw.columns.get_loc(f) for f in selected_features]

# 获取 SHAP 值子集
shap_values_sub = shap_values.values[:, selected_indices]
shap_data_sub = shap_values.data[:, selected_indices]

# 计算平均绝对 SHAP 值，在所选特征中选前10
mean_abs_shap = np.abs(shap_values_sub).mean(axis=0)
top10_idx_in_selected = np.argsort(mean_abs_shap)[-20:][::-1]

# 获取对应特征名
top10_features = [selected_features[i] for i in top10_idx_in_selected]

# 构造 SHAP Explanation 对象（仅前10个）
shap_values_top10 = shap.Explanation(
    values=shap_values_sub[:, top10_idx_in_selected],
    base_values=shap_values.base_values,
    data=shap_data_sub[:, top10_idx_in_selected],
    feature_names=top10_features
)

# 绘图
plt.figure(figsize=(8, 6))
shap.plots.beeswarm(
    shap_values_top10,
    max_display=20,
    color=custom_cmap
)
plt.tight_layout()
plt.show()


SHAP group plot

In [ ]:
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from matplotlib.colors import LinearSegmentedColormap


# =========================
# 1. 基本设置
# =========================
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"] = 14
plt.rcParams["axes.labelsize"] = 18
plt.rcParams["xtick.labelsize"] = 12
plt.rcParams["ytick.labelsize"] = 12

data_file = "Dataset_imputed_KNN.xlsx"
model_file = "final_CDHLQ.pkl"

target_col = "C"

dbp_mapping = {
    "TCM": "Class 1 (THMs)",
    "BDCM": "Class 1 (THMs)",
    "CDBM": "Class 1 (THMs)",

    "DCAA": "Class 2 (HAAs)",
    "TCAA": "Class 2 (HAAs)",
    "DBAA": "Class 2 (HAAs)",

    "DCAN": "Class 3 (HANs)",
    "BCAN": "Class 3 (HANs)",
    "DBAN": "Class 3 (HANs)",

    "1,1-DCP": "Class 4 (HKs)",
    "1,1,1-TCP": "Class 4 (HKs)"
}

selected_features = [
    "DOC", "UV254", "Br", "TN", "Chlorine dose", "Disinfection time",
    "Temperature", "pH", "C1", "C2", "LMW", "MMW", "HMW",
    "Es", "EHOMO", "ELUMO", "Egap"
]

custom_cmap = LinearSegmentedColormap.from_list(
    "custom_cmap",
    ["#867fba", "#86a4cb", "#efddaf", "#c11b24"]
)


In [ ]:
# =========================
# 2. 读取数据和模型
# =========================
df = pd.read_excel(data_file)

final_model = joblib.load(model_file)

preprocessor = final_model.named_steps["preprocess"]
model = final_model.named_steps["model"]


# =========================
# 3. 恢复与训练时一致的输入列
# =========================
model_input_cols = list(preprocessor.feature_names_in_)

X_model = df[model_input_cols].copy()
y = df[target_col].copy()

dbps_series = df["DBPs"].copy()


# =========================
# 4. 按之前完全一致的方式划分数据集
# =========================
X_train_raw, X_test_raw, y_train, y_test, dbps_train, dbps_test = train_test_split(
    X_model,
    y,
    dbps_series,
    test_size=0.2,
    random_state=90
)


# =========================
# 5. 使用已训练好的 preprocessor 转换训练集
# =========================
X_train_processed_array = preprocessor.transform(X_train_raw)

feature_names = preprocessor.get_feature_names_out()
feature_names = [
    name.replace("scale__", "").replace("remainder__", "")
    for name in feature_names
]

X_train_processed = pd.DataFrame(
    X_train_processed_array,
    columns=feature_names,
    index=X_train_raw.index
)

In [ ]:
# =========================
# 6. 计算整体 SHAP
# =========================
explainer = shap.TreeExplainer(model)
shap_values_all = explainer(X_train_processed)


# =========================
# 7. 筛选实际存在的特征
# =========================
selected_features_existing = [
    f for f in selected_features
    if f in X_train_processed.columns
]

selected_indices = [
    X_train_processed.columns.get_loc(f)
    for f in selected_features_existing
]


# =========================
# 8. 按 DBP 类别分别绘制 SHAP
# =========================
with pd.ExcelWriter("SHAP_by_DBP_Categories_CDHLQ.xlsx") as writer:

    for class_label in [
        "Class 1 (THMs)",
        "Class 2 (HAAs)",
        "Class 3 (HANs)",
        "Class 4 (HKs)"
    ]:

        target_dbps = [
            dbp for dbp, cls in dbp_mapping.items()
            if cls == class_label
        ]

        mask = dbps_train.isin(target_dbps).values

        if mask.sum() == 0:
            print(f"跳过 {class_label}: 训练集中没有对应样本")
            continue

        print(f"正在分析 {class_label}，样本量 = {mask.sum()}")

        current_shap_values = shap_values_all.values[mask][:, selected_indices]
        current_data = X_train_processed.values[mask][:, selected_indices]

        if hasattr(shap_values_all.base_values, "__len__"):
            current_base = shap_values_all.base_values[mask]
        else:
            current_base = shap_values_all.base_values

        mean_abs_shap = np.abs(current_shap_values).mean(axis=0)

        importance_df = pd.DataFrame({
            "Feature": selected_features_existing,
            "Mean(|SHAP|)": mean_abs_shap
        }).sort_values(
            by="Mean(|SHAP|)",
            ascending=False
        ).reset_index(drop=True)

        sheet_name = class_label.replace("Class ", "C").replace(" ", "_")[:31]
        importance_df.to_excel(writer, sheet_name=sheet_name, index=False)

        top_n = min(10, len(selected_features_existing))
        top_idx = np.argsort(mean_abs_shap)[-top_n:][::-1]

        top_features = [
            selected_features_existing[i]
            for i in top_idx
        ]

        shap_exp_sub = shap.Explanation(
            values=current_shap_values[:, top_idx],
            base_values=current_base,
            data=current_data[:, top_idx],
            feature_names=top_features
        )

        plt.figure(figsize=(8, 6))

        shap.plots.beeswarm(
            shap_exp_sub,
            max_display=top_n,
            color=custom_cmap,
            show=False
        )

        plt.title(class_label, fontsize=16, fontweight="bold")
        plt.tight_layout()

        fig_name = class_label.replace(" ", "_").replace("(", "").replace(")", "")
        plt.savefig(f"SHAP_{fig_name}_CDHLQ.png", dpi=600, bbox_inches="tight")
        plt.show()


print("✅ 按 DBP 类别分组的 SHAP 分析完成")
print("✅ 已导出：SHAP_by_DBP_Categories_CDHLQ.xlsx")

SHAP force plot

In [1]:
import pandas as pd

# =========================
# 1. 读取数据
# =========================
file_path = "Dataset_train_test_split.xlsx"
df = pd.read_excel(file_path)

split_column = "Dataset_split"
dbp_column = "DBPs"
target_column = "C"

# 检查必要列
required_columns = [split_column, dbp_column, target_column]
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise KeyError(f"数据中缺少必要列：{missing_columns}")

# =========================
# 2. 仅保留训练集样本
# =========================
training_df = df[
    df[split_column]
    .astype(str)
    .str.strip()
    .str.casefold()
    .eq("training")
].copy()

# 将浓度转换为数值，无法转换的内容记为缺失值
training_df[target_column] = pd.to_numeric(
    training_df[target_column],
    errors="coerce"
)

# 去除DBP名称或浓度缺失的样本
training_df = training_df.dropna(
    subset=[dbp_column, target_column]
)

if training_df.empty:
    raise ValueError("未找到Dataset_split为Training的有效样本。")

# =========================
# 3. 获取各DBP浓度最高和最低的样本
# =========================
print(f"训练集有效样本数：{len(training_df)}")
print(f"DBP物种数量：{training_df[dbp_column].nunique()}")
print("=" * 70)

# 按DBP在数据中的原始出现顺序输出
dbp_species = training_df[dbp_column].drop_duplicates()

for dbp_name in dbp_species:
    subset = training_df[
        training_df[dbp_column] == dbp_name
    ]

    highest_index = subset[target_column].idxmax()
    lowest_index = subset[target_column].idxmin()

    highest_value = df.loc[highest_index, target_column]
    lowest_value = df.loc[lowest_index, target_column]

    print(f"DBP：{dbp_name}")
    print(
        f"  最高浓度样本：Index = {highest_index}, "
        f"{target_column} = {highest_value}"
    )
    print(
        f"  最低浓度样本：Index = {lowest_index}, "
        f"{target_column} = {lowest_value}"
    )
    print("-" * 70)

训练集有效样本数：1896
DBP物种数量：11
DBP：DCAA
  最高浓度样本：Index = 1699, C = 90.67
  最低浓度样本：Index = 1689, C = 0.671
----------------------------------------------------------------------
DBP：BDCM
  最高浓度样本：Index = 1337, C = 88.044
  最低浓度样本：Index = 1663, C = 0.298
----------------------------------------------------------------------
DBP：DCAN
  最高浓度样本：Index = 1115, C = 23.873
  最低浓度样本：Index = 1422, C = 0.379
----------------------------------------------------------------------
DBP：TCM
  最高浓度样本：Index = 701, C = 99.701
  最低浓度样本：Index = 1477, C = 0.23
----------------------------------------------------------------------
DBP：TCAA
  最高浓度样本：Index = 39, C = 97.797
  最低浓度样本：Index = 244, C = 0.202
----------------------------------------------------------------------
DBP：CDBM
  最高浓度样本：Index = 949, C = 37.829
  最低浓度样本：Index = 1878, C = 0.153
----------------------------------------------------------------------
DBP：BCAN
  最高浓度样本：Index = 1654, C = 40.443
  最低浓度样本：Index = 549, C = 0.345
--------------------------

In [4]:
import pandas as pd
import numpy as np
import json
import joblib
import optuna

from xgboost import XGBRegressor

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


# =========================
# 1. 读取数据
# =========================
df0 = pd.read_excel("Dataset_imputed_KNN.xlsx")
df = df0.copy()

# 如需删除无关列，可取消注释
df.drop(columns=["Type", "DBPs","DOI", "Code"], inplace=True)


# =========================
# 2. 定义输入和输出
# =========================
target_col = "C"

X = df.drop(columns=[target_col])
y = df[target_col]


# =========================
# 3. 定义需要归一化的列
# =========================
numerical_cols_to_scale = [
    "DOC", "UV254", "Br", "TN", "Chlorine dose", "Disinfection time",
    "Temperature", "pH", "C1", "C2", "LMW", "MMW", "HMW",
    "Es", "EHOMO", "ELUMO", "Egap"
]


# =========================
# 4. 先划分训练集和测试集
# =========================
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=72
)




In [5]:
# =========================
# 5. 构建预处理器
# 仅用于说明模型结构，不重新拟合
# =========================
preprocessor = ColumnTransformer(
    transformers=[
        (
            "scale",
            StandardScaler(),
            numerical_cols_to_scale
        )
    ],
    remainder="passthrough"
)


# =========================
# 6. 读取已保存的配置文件
# =========================
with open(
    "scaler_columns.json",
    "r",
    encoding="utf-8"
) as file:
    saved_scaler_config = json.load(file)

with open(
    "best_params.json",
    "r",
    encoding="utf-8"
) as file:
    best_params = json.load(file)

# 兼容JSON中直接保存列表或使用字典保存的情况
if isinstance(saved_scaler_config, list):
    saved_scaler_columns = saved_scaler_config

elif isinstance(saved_scaler_config, dict):
    saved_scaler_columns = (
        saved_scaler_config.get("scaler_columns")
        or saved_scaler_config.get(
            "numerical_cols_to_scale"
        )
        or saved_scaler_config.get("columns")
    )

else:
    raise TypeError(
        "无法识别scaler_columns.json中的数据格式。"
    )

print("保存的归一化列：")
print(saved_scaler_columns)

print("\n保存的最优超参数：")
print(best_params)


# =========================
# 7. 加载完整的模型Pipeline
# =========================
final_pipeline = joblib.load(
    "final_CDHLQ.pkl"
)

if not isinstance(final_pipeline, Pipeline):
    raise TypeError(
        "final_CDHLQ.pkl不是sklearn Pipeline。"
        "如果该文件只保存了XGBRegressor，则还需要单独保存并加载"
        "已经拟合的ColumnTransformer或StandardScaler。"
    )

print("\n模型Pipeline包含以下步骤：")
print(list(final_pipeline.named_steps.keys()))


# =========================
# 8. 提取已拟合的预处理器和XGBoost模型
# =========================
if "preprocess" not in final_pipeline.named_steps:
    raise KeyError(
        "Pipeline中未找到名为'preprocess'的步骤。"
        f"现有步骤为：{list(final_pipeline.named_steps.keys())}"
    )

if "model" not in final_pipeline.named_steps:
    raise KeyError(
        "Pipeline中未找到名为'model'的步骤。"
        f"现有步骤为：{list(final_pipeline.named_steps.keys())}"
    )

fitted_preprocessor = (
    final_pipeline.named_steps["preprocess"]
)

best_model = (
    final_pipeline.named_steps["model"]
)

print("成功提取已拟合的预处理器和XGBoost模型")


# =========================
# 9. 检查归一化列是否一致
# =========================
if list(saved_scaler_columns) != list(
    numerical_cols_to_scale
):
    print(
        "⚠️ 警告：当前代码中的归一化列与"
        "scaler_columns.json不完全一致。"
    )
    print(
        "实际转换将使用final_CDHLQ.pkl中"
        "已经拟合的预处理器。"
    )


# =========================
# 10. 使用已拟合的预处理器转换数据
# 注意：这里只执行transform，不执行fit
# =========================
X_train_scaled_array = (
    fitted_preprocessor.transform(X_train_raw)
)

X_test_scaled_array = (
    fitted_preprocessor.transform(X_test_raw)
)

# 兼容稀疏矩阵输出
if hasattr(X_train_scaled_array, "toarray"):
    X_train_scaled_array = (
        X_train_scaled_array.toarray()
    )

if hasattr(X_test_scaled_array, "toarray"):
    X_test_scaled_array = (
        X_test_scaled_array.toarray()
    )


# =========================
# 11. 获取转换后的特征名称
# =========================
scaled_feature_names = (
    fitted_preprocessor.get_feature_names_out()
)

# 删除ColumnTransformer自动添加的前缀：
# scale__DOC -> DOC
# remainder__xxx -> xxx
cleaned_feature_names = [
    feature_name.split("__", 1)[-1]
    for feature_name in scaled_feature_names
]

if len(set(cleaned_feature_names)) != len(
    cleaned_feature_names
):
    raise ValueError(
        "删除ColumnTransformer前缀后出现了重复特征名称。"
    )


# =========================
# 12. 构建归一化后的DataFrame
# 关键：保留X_train_raw的原始索引
# =========================
X_train_scaled = pd.DataFrame(
    X_train_scaled_array,
    columns=cleaned_feature_names,
    index=X_train_raw.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled_array,
    columns=cleaned_feature_names,
    index=X_test_raw.index
)

print("\nX_train_scaled维度：")
print(X_train_scaled.shape)

print("\nX_test_scaled维度：")
print(X_test_scaled.shape)

print("\n转换后的特征顺序：")
print(X_train_scaled.columns.tolist())


# =========================
# 13. 检查模型与转换后数据的特征数
# =========================
if hasattr(best_model, "n_features_in_"):
    if (
        X_train_scaled.shape[1]
        != best_model.n_features_in_
    ):
        raise ValueError(
            "转换后的特征数量与模型输入数量不一致："
            f"{X_train_scaled.shape[1]} vs "
            f"{best_model.n_features_in_}"
        )


# =========================
# 14. 验证模型预测结果是否一致
# =========================
pipeline_predictions = (
    final_pipeline.predict(X_train_raw)
)

model_predictions = best_model.predict(
    X_train_scaled.to_numpy()
)

maximum_difference = np.max(
    np.abs(
        pipeline_predictions
        - model_predictions
    )
)

print(
    "\nPipeline预测与拆分后模型预测的最大差值："
    f"{maximum_difference:.10f}"
)

if maximum_difference < 1e-6:
    print("✅ 模型和归一化数据完全匹配")
else:
    print(
        "⚠️ 两种预测结果存在差异，请检查特征顺序。"
    )

保存的归一化列：
['DOC', 'UV254', 'Br', 'TN', 'Chlorine dose', 'Disinfection time', 'Temperature', 'pH', 'C1', 'C2', 'LMW', 'MMW', 'HMW', 'Es', 'EHOMO', 'ELUMO', 'Egap']

保存的最优超参数：
{'n_estimators': 1190, 'max_depth': 7, 'learning_rate': 0.021769858052516505, 'subsample': 0.8127204468230439, 'colsample_bytree': 0.9306358872951841, 'colsample_bylevel': 0.6120457392598307, 'colsample_bynode': 0.778270129945942, 'reg_alpha': 0.004038136910660049, 'reg_lambda': 0.007819555901792572, 'gamma': 0.43058356228734834, 'min_child_weight': 3, 'grow_policy': 'lossguide', 'max_leaves': 75, 'objective': 'reg:squarederror', 'random_state': 72, 'n_jobs': 1}

模型Pipeline包含以下步骤：
['preprocess', 'model']
成功提取已拟合的预处理器和XGBoost模型

X_train_scaled维度：
(1896, 17)

X_test_scaled维度：
(475, 17)

转换后的特征顺序：
['DOC', 'UV254', 'Br', 'TN', 'Chlorine dose', 'Disinfection time', 'Temperature', 'pH', 'C1', 'C2', 'LMW', 'MMW', 'HMW', 'Es', 'EHOMO', 'ELUMO', 'Egap']

Pipeline预测与拆分后模型预测的最大差值：0.0000000000
✅ 模型和归一化数据完全匹配


In [6]:
# ============================================================
# 15. 获取各DBP最高和最低浓度样本的SHAP结果
# 前置变量：
# df0、X_train_scaled、best_model
# ============================================================
import re
from pathlib import Path

import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# =========================
# 15.1 参数设置
# =========================
dbp_column = "DBPs"
target_column = "C"

# 单独显示前10个特征，其余特征合并
top_n = 10

plot_output_directory = Path(
    "SHAP_High_Low_Plots"
)
plot_output_directory.mkdir(
    parents=True,
    exist_ok=True
)

excel_output_path = (
    "SHAP_High_Low_By_DBP.xlsx"
)


# =========================
# 15.2 根据训练集索引提取DBP信息
# =========================
required_columns = [
    dbp_column,
    target_column
]

missing_columns = [
    column
    for column in required_columns
    if column not in df0.columns
]

if missing_columns:
    raise KeyError(
        f"df0中缺少必要列：{missing_columns}"
    )

# 通过原始索引确保DBP信息与归一化特征严格对应
training_info = df0.loc[
    X_train_scaled.index
].copy()

training_info[target_column] = pd.to_numeric(
    training_info[target_column],
    errors="coerce"
)

training_info = training_info.dropna(
    subset=[dbp_column, target_column]
)

if training_info.empty:
    raise ValueError(
        "训练集中没有有效的DBP名称或浓度数据。"
    )

print(
    f"用于SHAP分析的训练样本数："
    f"{len(training_info)}"
)

print(
    f"DBP物种数："
    f"{training_info[dbp_column].nunique()}"
)


# =========================
# 15.3 创建SHAP解释器
# =========================
explainer = shap.Explainer(
    best_model
)

print("✅ SHAP解释器创建完成")


# =========================
# 15.4 文件名处理函数
# =========================
def safe_file_name(name):
    """处理Windows文件名中的非法字符。"""

    return re.sub(
        r'[<>:"/\\|?*]',
        "_",
        str(name)
    )


def safe_sheet_name(name):
    """处理Excel工作表名称中的非法字符。"""

    cleaned_name = re.sub(
        r'[\\/*?:\[\]]',
        "_",
        str(name)
    )

    # Excel工作表名称不能超过31个字符
    return cleaned_name[:31]


# =========================
# 15.5 SHAP瀑布图绘制函数
# =========================
def plot_shap_waterfall(
    explanation,
    dbp_name,
    sample_type,
    sample_index,
    measured_concentration,
    predicted_concentration
):
    """
    绘制单个样本的SHAP瀑布图。

    max_display = top_n + 1：
    显示前10个特征，并将其余特征合并。
    """

    plt.rcParams.update({
        "font.size": 14,
        "axes.titlesize": 15,
        "axes.labelsize": 14
    })

    shap.plots.waterfall(
        explanation,
        max_display=top_n + 1,
        show=False
    )

    figure = plt.gcf()
    figure.set_size_inches(10, 8)

    plt.title(
        f"{dbp_name}: {sample_type} concentration\n"
        f"Index = {sample_index}, "
        f"Measured = {measured_concentration:.3f} μg/L, "
        f"Predicted = {predicted_concentration:.3f} μg/L",
        pad=20
    )

    dbp_file_name = safe_file_name(
        dbp_name
    )

    image_path = (
        plot_output_directory
        / (
            f"{dbp_file_name}_{sample_type}_"
            f"Index_{sample_index}.png"
        )
    )

    figure.savefig(
        image_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(figure)

    return str(image_path)


# =========================
# 15.6 将SHAP结果转换为表格
# =========================
def shap_explanation_to_dataframe(
    explanation,
    dbp_name,
    sample_type,
    sample_index,
    measured_concentration,
    predicted_concentration
):
    """整理单个样本的全部SHAP数据。"""

    shap_values = np.asarray(
        explanation.values
    ).reshape(-1)

    feature_values = np.asarray(
        explanation.data
    ).reshape(-1)

    base_value = float(
        np.asarray(
            explanation.base_values
        ).reshape(-1)[0]
    )

    result_df = pd.DataFrame({
        "DBP": dbp_name,
        "Sample_Type": sample_type,
        "Sample_Index": sample_index,
        "Measured_Concentration":
            measured_concentration,
        "Predicted_Concentration":
            predicted_concentration,
        "Base_Value": base_value,
        "Feature_Name":
            X_train_scaled.columns.tolist(),
        "Feature_Value_Normalized":
            feature_values,
        "SHAP_Value":
            shap_values,
        "Absolute_SHAP_Value":
            np.abs(shap_values)
    })

    # 按绝对SHAP值从高到低排列
    result_df = result_df.sort_values(
        by="Absolute_SHAP_Value",
        ascending=False
    ).reset_index(drop=True)

    result_df["Importance_Rank"] = (
        np.arange(
            1,
            len(result_df) + 1
        )
    )

    # 标记瀑布图中单独显示的前10个特征
    result_df["Displayed_in_plot"] = (
        result_df["Importance_Rank"]
        <= top_n
    )

    return result_df


# =========================
# 15.7 分析每种DBP的最高和最低浓度样本
# =========================
summary_records = []
dbp_result_tables = {}

dbp_species = (
    training_info[dbp_column]
    .drop_duplicates()
    .tolist()
)

for dbp_name in dbp_species:

    dbp_subset = training_info[
        training_info[dbp_column]
        == dbp_name
    ]

    highest_index = (
        dbp_subset[target_column]
        .idxmax()
    )

    lowest_index = (
        dbp_subset[target_column]
        .idxmin()
    )

    highest_concentration = float(
        dbp_subset.loc[
            highest_index,
            target_column
        ]
    )

    lowest_concentration = float(
        dbp_subset.loc[
            lowest_index,
            target_column
        ]
    )

    print("=" * 70)
    print(f"正在分析DBP：{dbp_name}")

    print(
        f"最高浓度样本：Index = {highest_index}, "
        f"C = {highest_concentration:.3f} μg/L"
    )

    print(
        f"最低浓度样本：Index = {lowest_index}, "
        f"C = {lowest_concentration:.3f} μg/L"
    )

    extreme_samples = [
        {
            "Sample_Type": "Highest",
            "Sample_Index": highest_index,
            "Concentration":
                highest_concentration
        },
        {
            "Sample_Type": "Lowest",
            "Sample_Index": lowest_index,
            "Concentration":
                lowest_concentration
        }
    ]

    current_dbp_results = []

    for sample_information in extreme_samples:

        sample_type = sample_information[
            "Sample_Type"
        ]

        sample_index = sample_information[
            "Sample_Index"
        ]

        measured_concentration = (
            sample_information[
                "Concentration"
            ]
        )

        # 根据原始索引提取归一化特征
        sample_features = (
            X_train_scaled.loc[
                [sample_index]
            ]
        )

        # 使用拆分出的XGBoost模型进行预测
        predicted_concentration = float(
            best_model.predict(
                sample_features.to_numpy()
            )[0]
        )

        # 计算该样本的SHAP Explanation
        sample_explanation = explainer(
            sample_features
        )[0]

        # 绘制SHAP瀑布图
        image_path = plot_shap_waterfall(
            explanation=sample_explanation,
            dbp_name=dbp_name,
            sample_type=sample_type,
            sample_index=sample_index,
            measured_concentration=
                measured_concentration,
            predicted_concentration=
                predicted_concentration
        )

        # 整理SHAP数据
        sample_shap_table = (
            shap_explanation_to_dataframe(
                explanation=
                    sample_explanation,
                dbp_name=dbp_name,
                sample_type=sample_type,
                sample_index=sample_index,
                measured_concentration=
                    measured_concentration,
                predicted_concentration=
                    predicted_concentration
            )
        )

        current_dbp_results.append(
            sample_shap_table
        )

        summary_records.append({
            "DBP": dbp_name,
            "Sample_Type": sample_type,
            "Sample_Index": sample_index,
            "Measured_Concentration":
                measured_concentration,
            "Predicted_Concentration":
                predicted_concentration,
            "Base_Value": float(
                np.asarray(
                    sample_explanation.base_values
                ).reshape(-1)[0]
            ),
            "Image_Path": image_path
        })

        print(
            f"{sample_type}样本SHAP图已保存："
            f"{image_path}"
        )

    # 合并同一种DBP的最高和最低浓度结果
    dbp_result_tables[dbp_name] = pd.concat(
        current_dbp_results,
        ignore_index=True
    )


# =========================
# 15.8 导出Excel
# 每种DBP对应一个工作表
# =========================
summary_df = pd.DataFrame(
    summary_records
)

with pd.ExcelWriter(
    excel_output_path,
    engine="openpyxl"
) as writer:

    # 导出汇总表
    summary_df.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    summary_worksheet = (
        writer.sheets["Summary"]
    )

    summary_worksheet.freeze_panes = "A2"
    summary_worksheet.auto_filter.ref = (
        summary_worksheet.dimensions
    )

    # 每种DBP导出到独立工作表
    for (
        dbp_name,
        result_table
    ) in dbp_result_tables.items():

        sheet_name = safe_sheet_name(
            dbp_name
        )

        result_table.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False
        )

        worksheet = writer.sheets[
            sheet_name
        ]

        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = (
            worksheet.dimensions
        )


print("=" * 70)
print("✅ 所有DBP的SHAP分析完成")

print(
    f"SHAP瀑布图保存目录："
    f"{plot_output_directory.resolve()}"
)

print(
    f"SHAP数据表："
    f"{Path(excel_output_path).resolve()}"
)

用于SHAP分析的训练样本数：1896
DBP物种数：11
✅ SHAP解释器创建完成
正在分析DBP：DCAA
最高浓度样本：Index = 1047, C = 90.670 μg/L
最低浓度样本：Index = 1259, C = 0.671 μg/L
Highest样本SHAP图已保存：SHAP_High_Low_Plots\DCAA_Highest_Index_1047.png
Lowest样本SHAP图已保存：SHAP_High_Low_Plots\DCAA_Lowest_Index_1259.png
正在分析DBP：BDCM
最高浓度样本：Index = 556, C = 88.044 μg/L
最低浓度样本：Index = 1146, C = 0.298 μg/L
Highest样本SHAP图已保存：SHAP_High_Low_Plots\BDCM_Highest_Index_556.png
Lowest样本SHAP图已保存：SHAP_High_Low_Plots\BDCM_Lowest_Index_1146.png
正在分析DBP：DCAN
最高浓度样本：Index = 107, C = 23.873 μg/L
最低浓度样本：Index = 364, C = 0.379 μg/L
Highest样本SHAP图已保存：SHAP_High_Low_Plots\DCAN_Highest_Index_107.png
Lowest样本SHAP图已保存：SHAP_High_Low_Plots\DCAN_Lowest_Index_364.png
正在分析DBP：TCM
最高浓度样本：Index = 489, C = 99.701 μg/L
最低浓度样本：Index = 1515, C = 0.230 μg/L
Highest样本SHAP图已保存：SHAP_High_Low_Plots\TCM_Highest_Index_489.png
Lowest样本SHAP图已保存：SHAP_High_Low_Plots\TCM_Lowest_Index_1515.png
正在分析DBP：TCAA
最高浓度样本：Index = 509, C = 97.797 μg/L
最低浓度样本：Index = 1265, C = 0.202 μg/L
Highest样本SHAP图已保存：